# Figures 5–7

**Deformation-dependent permeability**

This notebook evaluates the homogenized permeability under uniaxial stretch, simple shear, and pure shear loading while maintaining a prescribed macroscopic liquid-pressure gradient.

---

**Authors**

Haotian XIAO
Martin GENET

École Polytechnique, Palaiseau, France


## Imports

In [ ]:
from pathlib import Path
import math

import dolfin
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np
import Quiet_Microporoflow as qm
pv = qm.import_pyvista()  # = import pyvista as pv, without the VTK/tqdm start-up messages

import myPythonLibrary as mypy
import dolfin_mech as dmech

# Quieter output of the finite-element runs (messages only, results unchanged; see Quiet_Microporoflow.py):
# no gmsh information/progress lines, no ufl FutureWarning
qm.quiet_fe_output()

## Output directory

In [ ]:
# All results and figures of this notebook are written to the subdirectory Fig5-6-7/
# of the notebook directory (demos/).
RESULTS_DIR = Path("Fig5-6-7").resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("RESULTS_DIR =", qm.display_path(RESULTS_DIR))

## Deformation-loading function

In [ ]:
def run_deformation_mode(
    results_dir,
    mode,
    r0_list=(0.2,),
    probe_list=("gx", "gy"),
    perform_tests=0,
    beta=1.3,
):
    mode_to_folder = {
        "stretch-x": "Figure5_stretch_x",
        "stretch-y": "Figure6_stretch_y",
        "shear": "Figure7_shear",
        "pure-shear": "Pure_shear",
    }

    if mode not in mode_to_folder:
        raise ValueError(
            "mode must be 'stretch-x', 'stretch-y', "
            "'shear', or 'pure-shear'"
        )

    res_folder = Path(results_dir) / mode_to_folder[mode]
    res_folder.mkdir(parents=True, exist_ok=True)

    test = mypy.Test(
        res_folder=str(res_folder),
        perform_tests=perform_tests,
        stop_at_failure=1,
        clean_after_tests=0,
        tester_numpy_tolerance=1e-2,
    )

    material_parameters = {
        "alpha": 0.16,
        "gamma": 0.5,
        "c1": 0.2,
        "c2": 0.4,
        "kappa": 1e2,
        "eta": 1e-5,
    }

    dim = 2
    n_steps = 2
    pf = 0.0

    def run_one_probe(r0, probe):
        if probe == "gx":
            gx_ini_lst = [0.0, 0.01]
            gx_fin_lst = [0.01, 0.01]
            gy_ini_lst = [0.0, 0.0]
            gy_fin_lst = [0.0, 0.0]

        elif probe == "gy":
            gx_ini_lst = [0.0, 0.0]
            gx_fin_lst = [0.0, 0.0]
            gy_ini_lst = [0.0, 0.01]
            gy_fin_lst = [0.01, 0.01]

        else:
            raise ValueError("probe must be 'gx' or 'gy'")

        res_basename = f"Fig4-{mode}-r0={r0}-pf={pf}-{probe}"
        result_basename = res_folder / res_basename
        mesh_filebasename = res_folder / f"mesh-r0={r0}"

        load_params = {
            "solid": {},
            "liquid": {},
            "air": {},
        }

        for i in range(dim):
            for j in range(dim):
                load_params["solid"][f"sigma_bar_{i}{j}"] = 0.0

        load_params["liquid"]["pl_bar_ini_lst"] = [0.0] * n_steps
        load_params["liquid"]["pl_bar_fin_lst"] = [0.0] * n_steps

        load_params["liquid"]["grad_p_bar_x_ini_lst"] = gx_ini_lst
        load_params["liquid"]["grad_p_bar_x_fin_lst"] = gx_fin_lst
        load_params["liquid"]["grad_p_bar_y_ini_lst"] = gy_ini_lst
        load_params["liquid"]["grad_p_bar_y_fin_lst"] = gy_fin_lst

        load_params["liquid"]["Theta_in_ini_lst"] = [0.0] * n_steps
        load_params["liquid"]["Theta_in_fin_lst"] = [0.0] * n_steps
        load_params["liquid"]["Theta_out_ini_lst"] = [0.0] * n_steps
        load_params["liquid"]["Theta_out_fin_lst"] = [0.0] * n_steps

        if mode == "stretch-x":
            load_params["solid"]["U_bar_00_lst"] = [0.0, 0.3]

        elif mode == "stretch-y":
            load_params["solid"]["U_bar_11_lst"] = [0.0, 0.3]

        elif mode == "shear":
            load_params["solid"]["U_bar_00_lst"] = [0.0, 0.0]
            load_params["solid"]["U_bar_11_lst"] = [0.0, 0.0]
            load_params["solid"]["U_bar_01_lst"] = [0.0, 0.3]
            load_params["solid"]["U_bar_10_lst"] = [0.0, 0.0]

        elif mode == "pure-shear":
            load_params["solid"]["U_bar_00_lst"] = [
                0.0,
                beta - 1.0,
            ]
            load_params["solid"]["U_bar_11_lst"] = [
                0.0,
                1.0 / beta - 1.0,
            ]

        load_params["air"]["pf"] = pf
        load_params["air"]["pf_lst"] = [pf] * n_steps

        dmech.run_HollowBox_MicroPoroflow(
            dim=dim,
            mesh_params={
                "dim": dim,
                "xmin": 0.0,
                "ymin": -math.sqrt(3.0) / 2.0,
                "xmax": 1.0,
                "ymax": math.sqrt(3.0) / 2.0,
                "r0": r0,
                "l": 0.03,
                "hole_shape": "hex",
                "add_center_hole": True,
                "mesh_filebasename": str(mesh_filebasename),
            },
            mat_params={
                "skel": {
                    "parameters": material_parameters,
                    "scaling": "no",
                },
                "bulk": {
                    "parameters": material_parameters,
                    "scaling": "no",
                },
                "pore": {
                    "parameters": material_parameters,
                    "scaling": "no",
                },
            },
            flow_params={
                "k_l": dolfin.Constant(
                    (
                        (1e-6, 0.0),
                        (0.0, 1e-6),
                    )
                ),
                "use_kozeny_carman": False,
            },
            porosity_params={
                "type": "constant",
                "val": 0.3,
            },
            bcs="pbc",
            step_params={
                "n_steps": n_steps,
                "Deltat_lst": [1e-2, 1e-1],
                "dt_ini_lst": [1e-3, 1e-3],
                "dt_min_lst": [1e-4, 1e-4],
                "dt_max_lst": [5e-3, 5e-3],
            },
            load_params=load_params,
            res_basename=str(result_basename),
            verbose=0,
        )

        test.test(str(result_basename))

    for r0 in r0_list:
        for probe in probe_list:
            print(
                f"Running mode={mode}, "
                f"r0={r0}, "
                f"probe={probe}"
            )

            run_one_probe(
                r0=r0,
                probe=probe,
            )

    return res_folder

## Parameters

In [ ]:
r0_list = (0.072, 0.315, 0.41, 0.5)
mode_list = ("stretch-x", "stretch-y", "shear")
probe_list = ("gx", "gy")


## Computing response to deformation loading

In [ ]:
res_folders = {}

for mode in mode_list:
    res_folders[mode] = run_deformation_mode(
        results_dir=RESULTS_DIR,
        mode=mode,
        r0_list=r0_list,
        probe_list=probe_list,
    )

    print(f"Results for {mode} saved in: {qm.display_path(res_folders[mode])}")

## Plot Figures 5–7

In [ ]:
from Plot_Microporoflow import plot_principal_K_vs_U

def build_deformation_figure(
    mode,
    res_folder,
    r0_list,
):
    mode_to_xcomponent = {
        "stretch-x": "xx",
        "stretch-y": "yy",
        "shear": "xy",
        "pure-shear": "xx",
    }

    mode_to_figure_name = {
        "stretch-x": "Figure5_stretch_x.pdf",
        "stretch-y": "Figure6_stretch_y.pdf",
        "shear": "Figure7_shear.pdf",
        "pure-shear": "Figure7_pure_shear.pdf",
    }

    if mode not in mode_to_xcomponent:
        raise ValueError(
            "mode must be 'stretch-x', 'stretch-y', "
            "'shear', or 'pure-shear'"
        )

    figures_folder = RESULTS_DIR / "figures"
    figures_folder.mkdir(parents=True, exist_ok=True)

    figure_path = figures_folder / mode_to_figure_name[mode]

    plot_principal_K_vs_U(
        res_folder=str(res_folder),
        res_basename_prefix=f"Fig4-{mode}",
        r0_list=r0_list,
        pf_list=(0.0,),
        x_component=mode_to_xcomponent[mode],
        stream_density=0.6,
        stream_grid_n=500,
        stream_scale=1.0,
        add_stream_colorbar=True,
        save_name=str(figure_path),
        show_plot=True,
    )

    return figure_path


### Figure 5

In [ ]:
figure_path = build_deformation_figure(
    mode="stretch-x",
    res_folder=res_folders["stretch-x"],
    r0_list=r0_list,
)

print("Figure saved in:", qm.display_path(figure_path))

### Figure 6

In [ ]:
figure_path = build_deformation_figure(
    mode="stretch-y",
    res_folder=res_folders["stretch-y"],
    r0_list=r0_list,
)

### Figure 7

In [ ]:
figure_path = build_deformation_figure(
    mode="shear",
    res_folder=res_folders["shear"],
    r0_list=r0_list,
)